# Clinical Diagnostics CX Intelligence Platform

**Service Tier Recommendation Engine**

*Author: Aayush*

## Overview

This notebook takes **segment + propensity score + ARR** and produces per-customer service tier recommendations. 

## How to Use

Each numbered section below corresponds to one Databricks notebook cell. Run cells sequentially from top to bottom.

## What This Notebook Does

1. Joins **customer_360** + **customer_segments** + **customer_propensity**
2. Applies a rules-based recommendation engine producing one of:
   * **Platinum / Gold / Silver / Bronze** as the recommended tier
3. Compares current tier vs recommended tier and identifies misalignments
4. Quantifies financial exposure (ARR-weighted) of misaligned accounts
5. Writes results to `tier_recommendations` for the dashboard / exec summary


In [0]:
 #Config + load
 CATALOG = "cx"
SCHEMA_GOLD = "cx_gold"

import pandas as pd
import numpy as np

c360 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.customer_360").toPandas()
segs = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.customer_segments").toPandas()
prop = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.customer_propensity").toPandas()

df = c360.merge(segs, on="customer_id", how="left").merge(prop, on="customer_id", how="left")
print(f"Customer base: {len(df):,}")
print(f"Columns: {list(df.columns)}")

Customer base: 50,000
Columns: ['customer_id', 'customer_name', 'customer_type', 'region', 'country', 'service_tier', 'contract_start_date', 'annual_recurring_revenue_usd', 'employee_count', 'primary_contact_email', 'ingestion_timestamp', 'source_file', 'dq_flag_missing_email', 'dq_flag_missing_type', 'total_tickets', 'critical_tickets', 'escalated_tickets', 'avg_first_response_min', 'avg_resolution_hours', 'avg_post_ticket_csat', 'last_ticket_date', 'avg_monthly_test_volume', 'avg_uptime_pct', 'avg_monthly_errors', 'active_instruments', 'software_versions_used', 'latest_nps', 'latest_nps_bucket', 'latest_csat', 'latest_ces', 'latest_survey_date', 'total_survey_responses', 'avg_nps_lifetime', 'tenure_days', 'tenure_years', 'tickets_per_year', 'has_recent_critical', 'engagement_score', 'segment_id', 'segment_name', 'detractor_risk_score', 'risk_band']



## The Recommendation Logic

A customer's recommended tier is a function of three things:

1. **Customer value (ARR)** — how much revenue they represent
2. **Detractor risk** — how likely they are to be unhappy
3. **Segment** — their behavioral profile

### Tier Definitions

| Tier | Profile | Typical service |
|---|---|---|
| **Platinum** | High ARR + high risk OR Strategic Champion at any ARR | Dedicated CSM, quarterly business reviews, 24/7 priority support |
| **Gold** | Mid-high ARR + medium risk OR At-Risk High-Touch (any ARR) | Named support contact, 4-hour SLA, semi-annual reviews |
| **Silver** | Standard customers (most of the base) | Standard support, annual review |
| **Bronze** | Low engagement, dormant, or very low ARR | Self-service portal, light-touch outreach |

### The rule order matters
Rules are evaluated top-to-bottom; first match wins.

In [0]:
#Define recommendation logic
# ARR percentile thresholds (calculated from your data)
arr_p75 = df["annual_recurring_revenue_usd"].quantile(0.75)
arr_p50 = df["annual_recurring_revenue_usd"].quantile(0.50)
arr_p25 = df["annual_recurring_revenue_usd"].quantile(0.25)

print(f"ARR thresholds:")
print(f"  Top 25% (Platinum candidates):     ARR ≥ ${arr_p75:,.0f}")
print(f"  Middle 50% (Gold/Silver):           ${arr_p25:,.0f} – ${arr_p75:,.0f}")
print(f"  Bottom 25% (Bronze candidates):     ARR < ${arr_p25:,.0f}")


ARR thresholds:
  Top 25% (Platinum candidates):     ARR ≥ $75,999
  Middle 50% (Gold/Silver):           $17,202 – $75,999
  Bottom 25% (Bronze candidates):     ARR < $17,202


In [0]:
#Apply rules based engine
def recommend_tier(row):
    """
    Rules-based service tier recommendation.
    Returns: (recommended_tier, reason)
    """
    arr = row["annual_recurring_revenue_usd"]
    risk = row["detractor_risk_score"] if pd.notna(row["detractor_risk_score"]) else 0.5
    segment = row["segment_name"]

    # Rule 1: At-Risk High-Touch with high ARR → Platinum (white-glove)
    if segment == "At-Risk High-Touch" and arr >= arr_p50:
        return "Platinum", "High-value at-risk account requires dedicated intervention"

    # Rule 2: Any high-ARR customer with high risk → Platinum
    if arr >= arr_p75 and risk >= 0.40:
        return "Platinum", "Top-quartile ARR with elevated detractor risk"

    # Rule 3: Strategic Champions (any ARR) → Gold or higher
    if segment == "Strategic Champions":
        if arr >= arr_p75:
            return "Platinum", "Strategic Champion with top-quartile ARR — preserve at all costs"
        else:
            return "Gold", "Strategic Champion — proactive retention investment"

    # Rule 4: At-Risk High-Touch (any remaining ARR) → Gold
    if segment == "At-Risk High-Touch":
        return "Gold", "At-Risk profile — proactive support tier"

    # Rule 5: Dormant / Disengaged → Bronze (light touch)
    if segment == "Dormant / Disengaged":
        return "Bronze", "Low engagement — self-service tier with re-engagement campaigns"

    # Rule 6: High ARR Silent Majority → Gold (protect from silent churn)
    if segment == "Silent Majority" and arr >= arr_p75:
        return "Gold", "High-value silent customer — proactive outreach to prevent churn"

    # Rule 7: Low ARR + low engagement → Bronze
    if arr < arr_p25:
        return "Bronze", "Low-value account — efficient self-service tier"

    # Default: Silver
    return "Silver", "Standard service tier"


# Apply to all customers
df[["recommended_tier", "recommendation_reason"]] = df.apply(
    recommend_tier, axis=1, result_type="expand"
)

print("Recommended tier distribution:")
print(df["recommended_tier"].value_counts())


Recommended tier distribution:
recommended_tier
Gold        15860
Bronze      15040
Silver      11536
Platinum     7564
Name: count, dtype: int64


---

## Compare Current vs Recommended

How many customers are in the right tier today? How many need upgrade/downgrade?

In [0]:
#tier change analysis
tier_rank = {"Bronze": 1, "Silver": 2, "Gold": 3, "Platinum": 4}
df["current_tier_rank"] = df["service_tier"].map(tier_rank)
df["recommended_tier_rank"] = df["recommended_tier"].map(tier_rank)
df["tier_change"] = df["recommended_tier_rank"] - df["current_tier_rank"]

def label_change(diff):
    if diff > 0:
        return "Upgrade"
    elif diff < 0:
        return "Downgrade"
    else:
        return "No change"

df["change_action"] = df["tier_change"].apply(label_change)

print("Change action distribution:")
print(df["change_action"].value_counts())
print(f"\nMisalignment rate: {(df['change_action'] != 'No change').mean():.1%}")

Change action distribution:
change_action
Upgrade      27444
No change    13968
Downgrade     8588
Name: count, dtype: int64

Misalignment rate: 72.1%


---

## Financial Exposure of Misalignments

What's the ARR at stake in upgrades and downgrades?

In [0]:
#ARR weighted exposure
exposure = (
    df.groupby("change_action")
    .agg(
        customers=("customer_id", "count"),
        total_arr=("annual_recurring_revenue_usd", "sum"),
        avg_arr=("annual_recurring_revenue_usd", "mean"),
    )
    .round(0)
)
exposure["pct_of_base"] = (100 * exposure["customers"] / exposure["customers"].sum()).round(1)
print(exposure)

print(f"\nTotal ARR in upgrade candidates: ${df[df['change_action']=='Upgrade']['annual_recurring_revenue_usd'].sum():,.0f}")
print(f"Total ARR in downgrade candidates: ${df[df['change_action']=='Downgrade']['annual_recurring_revenue_usd'].sum():,.0f}")

               customers     total_arr  avg_arr  pct_of_base
change_action                                               
Downgrade           8588  3.515287e+08  40933.0         17.2
No change          13968  6.406532e+08  45866.0         27.9
Upgrade            27444  2.316192e+09  84397.0         54.9

Total ARR in upgrade candidates: $2,316,192,227
Total ARR in downgrade candidates: $351,528,706


---

## Tier Transition Matrix

Where customers are moving — current tier to recommended tier

In [0]:
#Transition Matrix
transition = pd.crosstab(
    df["service_tier"],
    df["recommended_tier"],
    margins=True,
    margins_name="Total",
)
print("Transition matrix (rows = current, columns = recommended):")
print(transition)


Transition matrix (rows = current, columns = recommended):
recommended_tier  Bronze   Gold  Platinum  Silver  Total
service_tier                                            
Bronze              8242   8702      4059    6370  27373
Gold                2288   2350      1175    1790   7603
Silver              4510   4808      2330    3376  15024
Total              15040  15860      7564   11536  50000


---

## Save Recommendations to Gold Layer

In [0]:
#Persist tier_recommendations table
output_cols = [
    "customer_id",
    "service_tier",
    "annual_recurring_revenue_usd",
    "segment_name",
    "detractor_risk_score",
    "risk_band",
    "recommended_tier",
    "recommendation_reason",
    "change_action",
]
recommendations_out = df[output_cols].copy()
recommendations_out["annual_recurring_revenue_usd"] = recommendations_out["annual_recurring_revenue_usd"].round(2)
recommendations_out["detractor_risk_score"] = recommendations_out["detractor_risk_score"].round(4)

spark.createDataFrame(recommendations_out).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.tier_recommendations"
)
print(f"✅ Saved {len(recommendations_out):,} tier recommendations to {CATALOG}.{SCHEMA_GOLD}.tier_recommendations")

✅ Saved 50,000 tier recommendations to cx.cx_gold.tier_recommendations


In [0]:
%sql
 --Sanity check 1: change distribution
 SELECT change_action, COUNT(*) AS customers,
        ROUND(SUM(annual_recurring_revenue_usd), 0) AS total_arr,
        ROUND(AVG(annual_recurring_revenue_usd), 0) AS avg_arr
 FROM cx.cx_gold.tier_recommendations
 GROUP BY change_action
 ORDER BY total_arr DESC

change_action,customers,total_arr,avg_arr
Upgrade,27444,2.316192227E9,84397.0
No change,13968,6.40653158E8,45866.0
Downgrade,8588,3.51528706E8,40933.0


In [0]:
%sql
--Sanity check 2: who's getting upgraded and why
 SELECT recommended_tier, segment_name, COUNT(*) AS customers,
        ROUND(SUM(annual_recurring_revenue_usd), 0) AS total_arr
 FROM cx.cx_gold.tier_recommendations
 WHERE change_action = 'Upgrade'
 GROUP BY recommended_tier, segment_name
 ORDER BY total_arr DESC


recommended_tier,segment_name,customers,total_arr
Gold,Silent Majority,4221,7.53444676E8
Platinum,At-Risk High-Touch,4275,4.92313594E8
Platinum,Strategic Champions,2185,3.79740532E8
Silver,Silent Majority,6370,2.53593221E8
Gold,Strategic Champions,5568,1.64585446E8
Platinum,Silent Majority,861,1.48816075E8
Gold,At-Risk High-Touch,3721,6.6378549E7
Platinum,Dormant / Disengaged,243,5.7320133E7


In [0]:
%sql
--Sanity check 3: top upgrade candidates by ARR
 SELECT customer_id, segment_name, service_tier AS current_tier,
        recommended_tier, ROUND(annual_recurring_revenue_usd, 0) AS arr,
        ROUND(detractor_risk_score, 3) AS risk_score,
        recommendation_reason
 FROM cx.cx_gold.tier_recommendations
 WHERE change_action = 'Upgrade'
 ORDER BY annual_recurring_revenue_usd DESC
 LIMIT 20

customer_id,segment_name,current_tier,recommended_tier,arr,risk_score,recommendation_reason
CUST_023774,At-Risk High-Touch,Gold,Platinum,5489382.0,0.36,High-value at-risk account requires dedicated intervention
CUST_039365,Silent Majority,Bronze,Gold,2202970.0,0.346,High-value silent customer — proactive outreach to prevent churn
CUST_016567,Silent Majority,Bronze,Platinum,2145098.0,0.445,Top-quartile ARR with elevated detractor risk
CUST_028506,Silent Majority,Bronze,Gold,2046727.0,0.381,High-value silent customer — proactive outreach to prevent churn
CUST_033540,Silent Majority,Bronze,Gold,1947359.0,0.377,High-value silent customer — proactive outreach to prevent churn
CUST_012503,Silent Majority,Gold,Platinum,1941960.0,0.423,Top-quartile ARR with elevated detractor risk
CUST_038726,Silent Majority,Bronze,Gold,1727647.0,0.381,High-value silent customer — proactive outreach to prevent churn
CUST_022841,Silent Majority,Bronze,Gold,1641606.0,0.375,High-value silent customer — proactive outreach to prevent churn
CUST_000006,At-Risk High-Touch,Silver,Platinum,1604158.0,0.371,High-value at-risk account requires dedicated intervention
CUST_038894,Silent Majority,Bronze,Gold,1548569.0,0.39,High-value silent customer — proactive outreach to prevent churn


In [0]:
%sql
--Sanity check 4: pilot scope — top 100 upgrades
 SELECT recommended_tier, COUNT(*) AS customers,
        ROUND(SUM(annual_recurring_revenue_usd), 0) AS pilot_arr
 FROM (
   SELECT * FROM cx.cx_gold.tier_recommendations
   WHERE change_action = 'Upgrade'
   ORDER BY annual_recurring_revenue_usd DESC
   LIMIT 100
 )
 GROUP BY recommended_tier

recommended_tier,customers,pilot_arr
Platinum,46,5.2567607E7
Gold,54,5.6672235E7
